In [1]:
# import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

# Load the Dataset
PROJECT_DIR = Path.cwd().resolve().parent
DATA_PATH = PROJECT_DIR / "data" / "processed" / "matchupDiff.csv"

assert DATA_PATH.exists(), f"Missing file: {DATA_PATH}"

matchupDiff =  pd.read_csv(DATA_PATH)
matchupDiff.head()

,Unnamed: 0,Season,DayNum,WTeamID,LTeamID,WScore,LScore,team1Id,team2Id,y,...,Raw Defensive Efficiency Rank_diff,Raw Offensive Efficiency_diff,Raw Offensive Efficiency Rank_diff,Raw Tempo_diff,Raw Tempo Rank_diff,Seed_diff,StlRate_diff,TOPct_diff,Tempo_diff,eFGPct_diff
0,0,2003,134,1421,1411,92,84,1411,1421,0,...,-165,1.6,-43,1.1,-36,0,-0.0095,-1.0055,1.1023,1.6965
1,1,2003,136,1112,1436,80,51,1112,1436,1,...,-34,9.0,-133,10.0,-263,-15,0.0061,-2.4229,9.9893,2.7429
2,2,2003,136,1113,1272,84,71,1113,1272,0,...,123,3.2,-42,-0.7,25,3,-0.0261,0.1265,-0.6498,1.9572
3,3,2003,136,1141,1166,79,73,1141,1166,0,...,159,-5.4,34,3.3,-100,5,-0.0200,5.7503,3.3585,-0.0999
4,4,2003,136,1143,1301,76,74,1143,1301,1,...,-39,-1.7,19,2.2,-96,-1,-0.0206,-1.3144,2.1439,-1.0445


In [2]:
# train / test split

train_df = matchupDiff[(matchupDiff["Season"] >= 2003) & (matchupDiff["Season"] <= 2019)]
test_df  = matchupDiff[(matchupDiff["Season"] >= 2021) & (matchupDiff["Season"] <= 2024)]

# separate features and target
X_train = train_df.drop(columns=["y"])
y_train = train_df["y"]

X_test = test_df.drop(columns=["y"])
y_test = test_df["y"]

In [3]:
# Week 4 - preprocessing validation checks

# Check shape
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (1097, 105)
test_df shape: (256, 105)


In [5]:
# Check for missing values
train_nulls = train_df.isnull().sum().sort_values(ascending=False)
test_nulls = test_df.isnull().sum().sort_values(ascending=False)
print("\nTrain missing values (top 20):")
print(train_nulls.head(20))
print("\nTest missing values (top 20):")
print(test_nulls.head(20))


Train missing values (top 20):
Unnamed: 0                       0
Pre-Tournament.DE_diff           0
RankFTRate_diff                  0
RankFTPct_diff                   0
RankFG3Rate_diff                 0
RankFG3Pct_diff                  0
RankFG2Pct_diff                  0
RankDefFT_diff                   0
RankDef3PtFG_diff                0
RankDef2PtFG_diff                0
RankDE_diff                      0
RankBlockPct_diff                0
RankAdjTempo_diff                0
RankAdjOE_diff                   0
RankAdjEM_diff                   0
RankAdjDE_diff                   0
RankARate_diff                   0
Pre-Tournament.Tempo_diff        0
Pre-Tournament.RankTempo_diff    0
Pre-Tournament.RankOE_diff       0
dtype: int64

Test missing values (top 20):
Active Coaching Length Index_diff    1
Unnamed: 0                           0
RankAdjTempo_diff                    0
RankFTRate_diff                      0
RankFTPct_diff                       0
RankFG3Rate_diff             

In [7]:
# Duplicate check
train_dups = train_df.duplicated().sum()
test_dups = test_df.duplicated().sum()
print(f"nDuplicate rows - train: {train_dups}, test: {test_dups}")

nDuplicate rows - train: 0, test: 0


In [ ]:
# Clear separation of predictors and target (x and y) & Test/Train split validation

target_col = "y"
assert target_col in train_df.columns and target_col in test_df.columns, "Target column 'y' missing."
X_train_full = train_df.drop(columns=[target_col])
y_train_full = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]
print("\nx_train_full shape:", X_train_full.shape, "| y_train_full shape:", y_train_full.shape)
print("x_test shape:", X_test.shape, "| y_test shape:", y_test.shape)


x_train_full shape: (1097, 104) | y_train_full shape: (1097,)
x_test shape: (256, 104) | y_test shape: (256,)


In [11]:
# Distribution summary and skewness checks
numeric_cols = X_train_full.select_dtypes(include=[np.number]).columns.tolist()
print("\nNumeric feature count:", len(numeric_cols))
desc = X_train_full[numeric_cols].describe().T
desc["skew"] = X_train_full[numeric_cols].skew(numeric_only=True)
desc["kurtosis"] = X_train_full[numeric_cols].kurtosis(numeric_only=True)
print("\nDistribution summary (first 15 numeric features):")
print(desc.head(15))
high_skew = desc.loc[desc["skew"].abs() > 1].sort_values("skew", ascending=False)
print(f"\nHighly skewed features (|skew| > 1): {high_skew.shape[0]}")
print(high_skew[["mean", "std", "min", "max", "skew"]].head(20))


Numeric feature count: 102

Distribution summary (first 15 numeric features):
                                    count         mean         std        min  \
Unnamed: 0                         1097.0   548.000000  316.820927     0.0000   
Season                             1097.0  2011.061076    4.907995  2003.0000   
DayNum                             1097.0   139.071103    4.242905   134.0000   
WTeamID                            1097.0  1295.095716  102.480677  1104.0000   
LTeamID                            1097.0  1295.820419  105.777818  1101.0000   
WScore                             1097.0    75.238833   10.738235    47.0000   
LScore                             1097.0    63.790337   10.395806    29.0000   
team1Id                            1097.0  1234.269827   84.749061  1101.0000   
team2Id                            1097.0  1356.646308   83.751189  1125.0000   
winnerTeamId                       1097.0  1295.095716  102.480677  1104.0000   
team1_Seed                    

In [13]:
# Quantitative outlier check

def iqr_outlier_counts(df, cols):
    counts = {}
    for c in cols:
        q1 = df[c].quantile(0.25)
        q3 = df[c].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0 or np.isnan(iqr):
            counts[c] = 0
            continue
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        counts[c] = ((df[c] < lower) | (df[c] > upper)).sum()
    return pd.Series(counts).sort_values(ascending=False)
def zscore_outlier_counts(df, cols, z_thresh=3.0):
    counts = {}
    for c in cols:
        s = df[c]
        std = s.std()
        if std == 0 or np.isnan(std):
            counts[c] = 0
            continue
        z = (s - s.mean()) / std
        counts[c] = (z.abs() > z_thresh).sum()
    return pd.Series(counts).sort_values(ascending=False)
iqr_counts = iqr_outlier_counts(X_train_full, numeric_cols)
z_counts = zscore_outlier_counts(X_train_full, numeric_cols, z_thresh=3.0)
print("\nTop 15 IQR outlier counts (training set predictors):")
print(iqr_counts.head(15))
print("\nTop 15 Z-score outlier counts (|z|>3, training set predictors)")
print(z_counts.head(15))


Top 15 IQR outlier counts (training set predictors):
DayNum                                     187
Active Coaching Length Index_diff          166
RankAdjEM_diff                             155
Pre-Tournament.RankAdjEM_diff              136
Adjusted Offensive Efficiency Rank_diff    112
RankAdjOE_diff                             112
Pre-Tournament.RankAdjDE_diff              106
RankAdjDE_diff                             103
Adjusted Defensive Efficiency Rank_diff    103
Pre-Tournament.RankAdjOE_diff              102
Pre-Tournament.RankOE_diff                  71
Raw Offensive Efficiency Rank_diff          68
RankOE_diff                                 68
AdjEM_diff                                  53
Pre-Tournament.RankDE_diff                  48
dtype: int64

Top 15 Z-score outlier counts (|z|>3, training set predictors)
DayNum                                     51
RankAdjEM_diff                             22
Pre-Tournament.RankAdjDE_diff              21
RankAdjDE_diff            